# Transformer input ablations: Ddc / full Yb / imaginary Yb / real Yb

Updated interface for `dl_normalized_datasets` v2. Set CASE=30, 39, or 57. The terminal runner and notebook use the same code. Run from this code folder.


In [1]:
#!rm -rf results

In [1]:
from pathlib import Path
import subprocess
import sys
import json
import csv

CODE_DIR = Path.cwd()  # Folder containing train_transformer_imlive.py and updated transformer files
DATA_ROOT = Path('/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/dataset').resolve()  # EDIT if needed
CASE = 30              # 30, 39, or 57
TARGET = 'dq'          # 'dq' or 'vmag'; one target type per run
MODELS = ['tri']  # 'axial', 'dq' (global attention), 'tri' (triangle-update model)
HEAD = 'residual_gate'  # 'plain' or 'residual_gate' (DQ only: exact Ddc zero gate + linear skip)
if TARGET != 'dq': HEAD = 'plain'
VARIANTS = ['ddc', 'ddc_yb', 'ddc_imag', 'ddc_real', 'ddc_imag_pinv']
PINV_RCOND = 1e-8  # Relative SVD cutoff on RAW Im(Yb)
SEEDS = [42]#, 123, 2026]
EPOCHS = 200
BATCH_SIZE = 64
D_MODEL = 96
DEPTH = 4
NUM_HEADS = 6
DROPOUT = 0.1
WARMUP_EPOCHS = 5
LOSS_ALPHA = 0.5       # 1.0 means pure MSE
DEVICE = 'cuda'
SMOKE = False         # True uses tiny subsets/two epochs; not a performance estimate
RESUME = False
AMP = True

DATA_DIR = DATA_ROOT / f'case{CASE}'
OUT_DIR = CODE_DIR / ('smoke_results' if SMOKE else 'results') / f'case{CASE}_{TARGET}_{HEAD}'
assert (CODE_DIR / 'train_transformer_imlive.py').is_file(), 'Set CODE_DIR to the extracted updated code folder'
for name in ['train', 'val', 'test']:
    assert (DATA_DIR / f'dataset_{name}.npz').is_file(), f'Missing {name} data'
print('Data:', DATA_DIR)
print('Results:', OUT_DIR)


Data: /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/dataset/case30
Results: /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Axial_transformer_triangle/results/case30_dq_residual_gate


In [2]:
command = [sys.executable, '-u', str(CODE_DIR / 'train_transformer_imlive.py'),
    '--data-dir', str(DATA_DIR), '--out-dir', str(OUT_DIR),
    '--models', *MODELS, '--variants', *VARIANTS,
    '--target', TARGET, '--head', HEAD, '--seeds', *map(str, SEEDS),
    '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE),
    '--loss-alpha', str(LOSS_ALPHA), '--device', DEVICE,
    '--d-model', str(D_MODEL), '--depth', str(DEPTH), '--num-heads', str(NUM_HEADS),
    '--dropout', str(DROPOUT), '--warmup-epochs', str(WARMUP_EPOCHS),
    '--pinv-rcond', str(PINV_RCOND)]
if SMOKE: command.append('--smoke')
if RESUME: command.append('--resume')
if not AMP: command.append('--no-amp')
print(' '.join(command))
subprocess.run(command, cwd=CODE_DIR, check=True)


/opt/tljh/user/bin/python -u /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Axial_transformer_triangle/train_transformer_imlive.py --data-dir /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/dataset/case30 --out-dir /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Axial_transformer_triangle/results/case30_dq_residual_gate --models axial tri --variants ddc_imag_pinv --target dq --head residual_gate --seeds 42 --epochs 200 --batch-size 64 --loss-alpha 0.5 --device cuda --d-model 96 --depth 4 --num-heads 6 --dropout 0.1 --warmup-epochs 5 --pinv-rcond 1e-08
PyTorch 2.10.0+cu128 device cuda
Preparing raw Im(Yb) pseudoinverse; fitting new feature normalization on train only.
Dataset(dq, X=(6880, 3, 30, 30), y=(6880, 1, 30, 30)) 
 Dataset(dq, X=(1474, 3, 30, 30), y=(1474, 1, 30, 30))

axial ddc_imag_pinv seed=42: 901,155 parameters; AMP=True
epoch=  1 loss=0.13503 val_R2=0.8738004611696779 val_RMSE=1.516440 (13.7s)
epoch=  2 loss=0.13

CompletedProcess(args=['/opt/tljh/user/bin/python', '-u', '/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Axial_transformer_triangle/train_transformer_imlive.py', '--data-dir', '/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/dataset/case30', '--out-dir', '/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Axial_transformer_triangle/results/case30_dq_residual_gate', '--models', 'axial', 'tri', '--variants', 'ddc_imag_pinv', '--target', 'dq', '--head', 'residual_gate', '--seeds', '42', '--epochs', '200', '--batch-size', '64', '--loss-alpha', '0.5', '--device', 'cuda', '--d-model', '96', '--depth', '4', '--num-heads', '6', '--dropout', '0.1', '--warmup-epochs', '5', '--pinv-rcond', '1e-08'], returncode=0)

In [3]:
with (OUT_DIR / 'summary.json').open() as f:
    summary = json.load(f)
print('Metrics use original units:', summary['protocol']['metric_units'])
for model, variants in summary['aggregate'].items():
    for variant, splits in variants.items():
        print(model, variant, 'test:', splits['test'])
print('Training-mean baseline:', summary['training_mean_baseline'])
print('Reference baselines (Ddc-only):', summary.get('reference_baselines'))
print('Individual runs:', OUT_DIR / 'comparison.csv')


Metrics use original units: raw DQ
axial ddc_imag_pinv test: {'r2': {'mean': 0.8784811737080054, 'std_across_seeds': None}, 'rmse': {'mean': 1.5023124687684948, 'std_across_seeds': None}, 'mae': {'mean': 0.6107780160369681, 'std_across_seeds': None}, 'nz_r2': {'mean': 0.8478537538970974, 'std_across_seeds': None}, 'nz_rmse': {'mean': 2.032239982363619, 'std_across_seeds': None}, 'nz_mae': {'mean': 1.1176679837210486, 'std_across_seeds': None}}
tri ddc_imag_pinv test: {'r2': {'mean': 0.8801855288034492, 'std_across_seeds': None}, 'rmse': {'mean': 1.4917399696321334, 'std_across_seeds': None}, 'mae': {'mean': 0.6058965914776127, 'std_across_seeds': None}, 'nz_r2': {'mean': 0.8499876720538759, 'std_across_seeds': None}, 'nz_rmse': {'mean': 2.0179381271194647, 'std_across_seeds': None}, 'nz_mae': {'mean': 1.1087354220933818, 'std_across_seeds': None}}
Training-mean baseline: {'train': {'r2': 8.881784197001252e-16, 'rmse': 4.287195813855575, 'mae': 2.740145110263725}, 'validation': {'r2': -

## Notes

All four variants share the same fitted training normalization, parameter count within each architecture, initial weights per seed, and training protocol. Excluded slots are masked after normalization. Test data is evaluated only after validation checkpoint selection. Do not apply a second scaler or sigmoid to standardized targets. The saved predictions and R2/RMSE/MAE are in original units. Axial attention is the default. Global attention over B*B cells has much higher memory use; reduce batch size if selecting dq. See README.md for inference and recovery.
